## 1. Granger Causality

In [ ]:
import pandas as pd
from pathlib import Path

### 1.1 Consolidated Results

In [ ]:
def load_all_csvs(directory_path):
    data_dir = Path(directory_path)
    csv_files = list(data_dir.glob("*.csv"))

    if not csv_files:
        return None

    all_dataframes = []

    for file in csv_files:
        try:
            df = pd.read_csv(file)
            all_dataframes.append(df)
        except Exception as e:
            print(f"Error loading {file.name}: {e}")

    if all_dataframes:
        return pd.concat(all_dataframes, ignore_index=True)
    return None

file_dir = "../../results/01_granger_causality"
GC_master = load_all_csvs(file_dir)

In [ ]:
# Bringing the "event_date" into a standardized format
GC_master["event_date"] = pd.to_datetime(GC_master["event_date"], format = "mixed")
GC_master["event_date"] = GC_master["event_date"].dt.strftime("%Y-%m-%d")
GC_master['relationship'] = GC_master['source_market'] + ' -> ' + GC_master['target_market']

In [ ]:
GC_master

,event_date,contract,source_market,target_market,lag_minutes,f_stat,p_value,coef,coef_p_value,is_sig_5pct,relationship
0,2026-01-28,0bp,CME,PM,1,0.7820,0.376518,0.0028,0.376518,False,CME -> PM
1,2026-01-28,0bp,PM,CME,1,1.3075,0.252844,-0.0022,0.252844,False,PM -> CME
2,2026-01-28,0bp,CME,PM,5,18.7266,0.000015,0.0099,0.000015,True,CME -> PM
3,2026-01-28,0bp,PM,CME,5,1.6571,0.197990,-0.0016,0.197990,False,PM -> CME
4,2026-01-28,0bp,CME,PM,30,22.9150,0.000002,0.0063,0.000002,True,CME -> PM
...,...,...,...,...,...,...,...,...,...,...,...
811,2024-12-18,25bp_dec,KAL,PM,5,6.9863,0.008215,0.0037,0.008215,True,KAL -> PM
812,2024-12-18,25bp_dec,PM,KAL,30,0.0745,0.784845,-0.0009,0.784845,False,PM -> KAL
813,2024-12-18,25bp_dec,KAL,PM,30,29.6724,0.000000,0.0039,0.000000,True,KAL -> PM
814,2024-12-18,25bp_dec,PM,KAL,60,7.1230,0.007612,0.0070,0.007612,True,PM -> KAL


In [ ]:
# Function to calculate conditional stats (only for significant tests)
def get_conditional_stats(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats
    grouped = significant_data.groupby('relationship')[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# Process 5% Significance
df_5pct = GC_master.groupby('relationship')['is_sig_5pct'].agg(['sum', 'count', 'mean'])
df_5pct = df_5pct.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# Get avg coefficient and std for 5% significant results
stats_5pct = get_conditional_stats(GC_master, 'is_sig_5pct', 'coef') # Replace 'granger_coeff' with your actual column name
df_5pct = df_5pct.join(stats_5pct)

# Sort and display
df_5pct = df_5pct.sort_values(by='significance_rate_5pct', ascending=False)

# Ordering the data in the desired manner before printing
column_order_5pct = [
    'total_tests',
    'significant_count_5pct',
    'significance_rate_5pct',
    'avg_coeff',
    'std_coeff'
]

df_5pct = df_5pct[column_order_5pct]

print(df_5pct)

              total_tests  significant_count_5pct  significance_rate_5pct  \
relationship                                                                
CME -> KAL            136                     106                0.779412   
CME -> PM             136                      99                0.727941   
KAL -> PM             136                      96                0.705882   
PM -> KAL             136                      95                0.698529   
KAL -> CME            136                      67                0.492647   
PM -> CME             136                      64                0.470588   

              avg_coeff  std_coeff  
relationship                        
CME -> KAL     0.014661   0.024740  
CME -> PM      0.015188   0.017652  
KAL -> PM      0.015485   0.022909  
PM -> KAL      0.016471   0.016506  
KAL -> CME    -0.000570   0.057723  
PM -> CME      0.005830   0.020982  


### 1.2 Differentiated by lag

In [ ]:
# Function to calculate conditional stats (only for significant tests)
def get_conditional_stats(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats
    grouped = significant_data.groupby(['relationship',"lag_minutes"])[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# 1. Process 5% Significance
df_5pct = GC_master.groupby(['relationship',"lag_minutes"])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
df_5pct = df_5pct.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# Get avg coefficient and std for 5% significant results
stats_5pct = get_conditional_stats(GC_master, 'is_sig_5pct', 'coef') # Replace 'granger_coeff' with your actual column name
df_5pct = df_5pct.join(stats_5pct)

# Sort and display
df_5pct = df_5pct.sort_values(by=['lag_minutes','significance_rate_5pct'], ascending=[True,False])

# Ordering the data in the desired manner before printing
column_order_5pct = [
    'total_tests',
    'significant_count_5pct',
    'significance_rate_5pct',
    'avg_coeff',
    'std_coeff'
]

df_5pct = df_5pct[column_order_5pct]

print(df_5pct)

                          total_tests  significant_count_5pct  \
relationship lag_minutes                                        
CME -> KAL   1                     34                      22   
CME -> PM    1                     34                      20   
KAL -> PM    1                     34                      17   
PM -> KAL    1                     34                      17   
KAL -> CME   1                     34                      10   
PM -> CME    1                     34                      10   
CME -> KAL   5                     34                      24   
CME -> PM    5                     34                      24   
KAL -> PM    5                     34                      23   
PM -> KAL    5                     34                      23   
KAL -> CME   5                     34                      16   
PM -> CME    5                     34                      11   
CME -> KAL   30                    34                      27   
KAL -> PM    30          

### 1.3 Differentiated at volume

In [ ]:
volume_df = pd.read_csv("../../data/processed/Volume/volume.csv")
volume_df["tritile_CME"] =  pd.qcut(volume_df["CME"], q = 3, labels = ["low","medium","high"])
volume_df["tritile_PM"] = pd.qcut(volume_df["PM"], q =3, labels = ["low","medium","high"] )
volume_df["tritile_Kalshi"] = pd.qcut(volume_df["Kalshi"],q=3, labels =["low", "medium","high"])

volume_info = volume_df[["event_date","contract","tritile_CME","tritile_PM","tritile_Kalshi"]]
GC_merged = pd.merge(GC_master,volume_info, on = ["event_date","contract"], how = "left")

In [ ]:
GC_merged

,event_date,contract,source_market,target_market,lag_minutes,f_stat,p_value,coef,coef_p_value,is_sig_5pct,is_sig_1pct,relationship,tritile_CME,tritile_PM,tritile_Kalshi
0,2024-01-31,0bp,CME,PM,1,0.0000,0.999997,-0.0000,0.999996,False,False,CME -> PM,medium,low,low
1,2024-01-31,0bp,PM,CME,1,0.0000,0.999996,-0.0000,0.999996,False,False,PM -> CME,medium,low,low
2,2024-01-31,0bp,CME,PM,5,3.9333,0.047346,0.0152,0.047346,True,False,CME -> PM,medium,low,low
3,2024-01-31,0bp,PM,CME,5,0.0000,0.997807,0.0000,0.997807,False,False,PM -> CME,medium,low,low
4,2024-01-31,0bp,CME,PM,30,7.4968,0.006184,0.0114,0.006184,True,True,CME -> PM,medium,low,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811,2026-04-29,25bp_dec,KAL,PM,5,4.5253,0.033399,-0.0027,0.033399,True,False,KAL -> PM,medium,high,medium
812,2026-04-29,25bp_dec,PM,KAL,30,56.2575,0.000000,0.0059,0.000000,True,True,PM -> KAL,medium,high,medium
813,2026-04-29,25bp_dec,KAL,PM,30,2.8779,0.089807,0.0011,0.089807,False,False,KAL -> PM,medium,high,medium
814,2026-04-29,25bp_dec,PM,KAL,60,73.4200,0.000000,0.0053,0.000000,True,True,PM -> KAL,medium,high,medium


##### 1.3.1 CME

In [ ]:
CME_vol_df = GC_merged[(GC_merged["source_market"] == "CME")]

In [ ]:
# 1. Update the function to handle the new grouping level
def get_conditional_stats_tritle(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats by relationship AND tritle
    grouped = significant_data.groupby(['relationship', 'tritile_CME'])[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# 2. Re-calculate the main summary (this matches your existing code)
CME_source_vol_95 = CME_vol_df.groupby(['relationship','tritile_CME'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
CME_source_vol_95 = CME_source_vol_95.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# 3. Get avg coefficient and std for 5% significant results
# Make sure 'coef' is the correct column name in your DataFrame
stats_95 = get_conditional_stats_tritle(CME_vol_df, 'is_sig_5pct', 'coef')

# 4. Join them together
CME_source_vol_95 = CME_source_vol_95.join(stats_95)

CME_source_vol_95 = CME_source_vol_95[["total_tests","significant_count_5pct","significance_rate_5pct","avg_coeff","std_coeff"]]

# 5. Display the final result
print(CME_source_vol_95)

                          total_tests  significant_count_5pct  \
relationship tritile_CME                                        
CME -> KAL   low                   48                      34   
             medium                40                      27   
             high                  48                      45   
CME -> PM    low                   48                      21   
             medium                40                      33   
             high                  48                      45   

                          significance_rate_5pct  avg_coeff  std_coeff  
relationship tritile_CME                                                
CME -> KAL   low                        0.708333   0.006315   0.009348  
             medium                     0.675000   0.011022   0.012047  
             high                       0.937500   0.023151   0.034258  
CME -> PM    low                        0.437500   0.008414   0.012630  
             medium                     0

/tmp/ipykernel_3348/536042657.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  CME_source_vol_95 = CME_vol_df.groupby(['relationship','tritile_CME'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
/tmp/ipykernel_3348/536042657.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = significant_data.groupby(['relationship', 'tritile_CME'])[coeff_col].agg(['mean', 'std'])


##### 1.3.2 Polymarket

In [ ]:
pm_vol_df = GC_merged[GC_merged["source_market"] == "PM"]

In [ ]:
# 1. Update the function to handle the new grouping level
def get_conditional_stats_tritle(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats by relationship AND tritle
    grouped = significant_data.groupby(['relationship', 'tritile_PM'])[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# 2. Re-calculate the main summary (this matches your existing code)
pm_source_vol_95 = pm_vol_df.groupby(['relationship','tritile_PM'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
pm_source_vol_95 = pm_source_vol_95.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# 3. Get avg coefficient and std for 5% significant results
# Make sure 'coef' is the correct column name in your DataFrame
stats_95 = get_conditional_stats_tritle(pm_vol_df, 'is_sig_5pct', 'coef')

# 4. Join them together
pm_source_vol_95 = pm_source_vol_95.join(stats_95)

pm_source_vol_95 = pm_source_vol_95[["total_tests","significant_count_5pct","significance_rate_5pct","avg_coeff","std_coeff"]]

# 5. Display the final result
print(pm_source_vol_95)

                         total_tests  significant_count_5pct  \
relationship tritile_PM                                        
PM -> CME    low                  44                      12   
             medium               44                      24   
             high                 48                      28   
PM -> KAL    low                  44                      17   
             medium               44                      33   
             high                 48                      45   

                         significance_rate_5pct  avg_coeff  std_coeff  
relationship tritile_PM                                                
PM -> CME    low                       0.272727  -0.004400   0.036351  
             medium                    0.545455   0.004729   0.017704  
             high                      0.583333   0.011157   0.012034  
PM -> KAL    low                       0.386364   0.007312   0.014086  
             medium                    0.750000   0.022

/tmp/ipykernel_3348/1824585231.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pm_source_vol_95 = pm_vol_df.groupby(['relationship','tritile_PM'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
/tmp/ipykernel_3348/1824585231.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = significant_data.groupby(['relationship', 'tritile_PM'])[coeff_col].agg(['mean', 'std'])


#### 1.3.3 Kalshi

In [ ]:
kalshi_vol_df = GC_merged[GC_merged["source_market"] == "KAL"]

In [ ]:
# 1. Update the function to handle the new grouping level
def get_conditional_stats_tritle(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats by relationship AND tritle
    grouped = significant_data.groupby(['relationship', 'tritile_Kalshi'])[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# 2. Re-calculate the main summary (this matches your existing code)
kalshi_source_vol_95 = kalshi_vol_df.groupby(['relationship','tritile_Kalshi'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
kalshi_source_vol_95 = kalshi_source_vol_95.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# 3. Get avg coefficient and std for 5% significant results
# Make sure 'coef' is the correct column name in your DataFrame
stats_95 = get_conditional_stats_tritle(kalshi_vol_df, 'is_sig_5pct', 'coef')

# 4. Join them together
kalshi_source_vol_95 = kalshi_source_vol_95.join(stats_95)

kalshi_source_vol_95 = kalshi_source_vol_95[["total_tests","significant_count_5pct","significance_rate_5pct","avg_coeff","std_coeff"]]

# 5. Display the final result
print(kalshi_source_vol_95)

                             total_tests  significant_count_5pct  \
relationship tritile_Kalshi                                        
KAL -> CME   low                      48                      26   
             medium                   40                      11   
             high                     48                      30   
KAL -> PM    low                      48                      26   
             medium                   40                      30   
             high                     48                      40   

                             significance_rate_5pct  avg_coeff  std_coeff  
relationship tritile_Kalshi                                                
KAL -> CME   low                           0.541667  -0.015423   0.089306  
             medium                        0.275000   0.005836   0.006995  
             high                          0.625000   0.009953   0.019050  
KAL -> PM    low                           0.541667   0.009065   0.031109  

/tmp/ipykernel_3348/1182277951.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  kalshi_source_vol_95 = kalshi_vol_df.groupby(['relationship','tritile_Kalshi'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
/tmp/ipykernel_3348/1182277951.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = significant_data.groupby(['relationship', 'tritile_Kalshi'])[coeff_col].agg(['mean', 'std'])
